# SI4006 · M1 — Fine-tuning con LoRA
## Clasificación del nivel de desperdicio alimentario

**Proyecto:** Food Waste / Restaurants  
**Familia:** Encoder-only  
**Modelo base:** `distilbert-base-uncased`  
**Tarea:** clasificación multiclase: `LOW`, `MEDIUM`, `HIGH`

> Esta primera sección conserva la lógica del notebook de S03: **setup → preparación → elección del modelo → experimento → evaluación**.
>
> En M1 añadimos el flujo requerido: **dataset del dominio → baseline → LoRA → Trainer → comparación**.


## 0 · Setup

El notebook debe correr en **Google Colab con GPU T4**.

Siguiendo el notebook de S03, evitamos fijar versiones de `torch` o `transformers` para no generar conflictos con el entorno de Colab. Instalamos solamente las librerías que necesitamos para M1.


In [ ]:
# Instalación mínima para M1.
# No fijamos versiones de torch/transformers: usamos las versiones actuales de Colab.
%pip install -q -U peft datasets accelerate

import os
import random
import numpy as np
import pandas as pd
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("torch:", torch.__version__)
print("device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 1 · Carga del dataset

El dataset es `food_wastage_data.csv`.

En Colab, suban el archivo al entorno o móntenlo desde Drive. Para una entrega reproducible, se recomienda guardar el CSV en el repositorio, dentro de una carpeta como `data/`, y cambiar `DATA_PATH` si es necesario.


In [ ]:
# Si el CSV está en la misma carpeta del notebook:
DATA_PATH = "food_wastage_data.csv"

# Alternativa recomendada para el repositorio:
# DATA_PATH = "data/food_wastage_data.csv"

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
display(df.head())
print("\nColumnas:")
print(df.columns.tolist())


In [ ]:
# Diagnóstico básico del dataset
print("Valores faltantes:")
display(df.isna().sum().to_frame("missing"))

print("\nTipos:")
display(df.dtypes.to_frame("dtype"))

print("\nTarget original:")
display(df["Wastage Food Amount"].describe())

print("\nCategorías:")
for col in df.select_dtypes(include="object").columns:
    print(f"\n{col}")
    print(df[col].value_counts())


## 2 · De regresión a clasificación

`Wastage Food Amount` es originalmente una variable numérica. Para M1 la convertimos en tres clases.

Los terciles empíricos se encuentran aproximadamente en **21.33** y **35**. Como el target es discreto y hay observaciones repetidas en 35, no dividimos un mismo valor entre dos clases. Usamos los cortes enteros:

- `LOW`: `<= 21`
- `MEDIUM`: `22–34`
- `HIGH`: `>= 35`

Esto produce clases aproximadamente balanceadas.


In [ ]:
# Crear las etiquetas de clasificación
def make_label(waste):
    if waste <= 21:
        return "LOW"
    elif waste <= 34:
        return "MEDIUM"
    else:
        return "HIGH"

df["label"] = df["Wastage Food Amount"].apply(make_label)

label_order = ["LOW", "MEDIUM", "HIGH"]
label_counts = df["label"].value_counts().reindex(label_order)

print("Distribución de clases:")
display(pd.DataFrame({
    "count": label_counts,
    "percentage": (label_counts / len(df) * 100).round(2)
}))

assert df["label"].notna().all()
assert df["label"].nunique() == 3


### 2.1 · Comprobación de leakage

`Wastage Food Amount` es el target original. **No puede entrar en el texto de entrada.**

El modelo verá solamente las características disponibles antes de conocer el desperdicio.


In [ ]:
FEATURE_COLUMNS = [
    "Type of Food",
    "Number of Guests",
    "Event Type",
    "Quantity of Food",
    "Storage Conditions",
    "Purchase History",
    "Seasonality",
    "Preparation Method",
    "Geographical Location",
    "Pricing",
]

TARGET_COLUMN = "Wastage Food Amount"

assert TARGET_COLUMN not in FEATURE_COLUMNS
print("OK: Wastage Food Amount no se utiliza como feature.")


## 3 · Convertir cada fila tabular en texto

DistilBERT recibe texto. Por eso convertimos las variables de cada evento en una descripción estructurada en inglés.

Esta transformación **no agrega información nueva**: solamente cambia la representación de las variables para que el encoder pueda procesarlas como texto.


In [ ]:
def row_to_text(row):
    return (
        f"The food type is {str(row['Type of Food']).lower()}. "
        f"There are {row['Number of Guests']} guests. "
        f"The event type is {str(row['Event Type']).lower()}. "
        f"The quantity of food is {row['Quantity of Food']}. "
        f"The storage condition is {str(row['Storage Conditions']).lower()}. "
        f"The purchase history is {str(row['Purchase History']).lower()}. "
        f"The seasonality is {str(row['Seasonality']).lower()}. "
        f"The preparation method is {str(row['Preparation Method']).lower()}. "
        f"The geographical location is {str(row['Geographical Location']).lower()}. "
        f"The pricing level is {str(row['Pricing']).lower()}."
    )

df["text"] = df.apply(row_to_text, axis=1)

print(df.loc[0, "text"])
print("\nLabel:", df.loc[0, "label"])


## 4 · Train / validation split

Usamos un split **80/20 estratificado**. La estratificación mantiene aproximadamente la misma proporción de `LOW`, `MEDIUM` y `HIGH` en ambos conjuntos.


In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df[["text", "label"]],
    test_size=0.20,
    random_state=SEED,
    stratify=df["label"],
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)

print("\nTrain distribution:")
display(train_df["label"].value_counts(normalize=True).sort_index())

print("\nValidation distribution:")
display(val_df["label"].value_counts(normalize=True).sort_index())


## 5 · Carga del tokenizer y modelo base

### ¿Por qué encoder-only?

Las diapositivas de S03 presentan los encoder-only como modelos de atención bidireccional adecuados para **clasificar y extraer**. La regla práctica de la sesión es:

> clasifican/extraen → encoder → DistilBERT

Nuestro input se entiende completo antes de asignarlo a una categoría, por lo que esta familia coincide con la tarea.


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

BASE_MODEL = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

label2id = {"LOW": 0, "MEDIUM": 1, "HIGH": 2}
id2label = {v: k for k, v in label2id.items()}

print("Modelo base:", BASE_MODEL)
print("Labels:", label2id)

# Consejo de la clase: revisar cómo tokeniza el vocabulario del dominio.
sample = train_df.loc[0, "text"]
tokens = tokenizer.tokenize(sample)

print("\nEjemplo de entrada:")
print(sample)
print("\nPrimeros tokens:")
print(tokens[:40])


## 6 · Baseline

La asignación pide una comparación explícita. El baseline principal será el **mismo modelo base sin fine-tuning**, usado zero-shot mediante su objetivo de masked language modeling.

Le pedimos completar:

> `The food waste level is [MASK].`

y comparamos las probabilidades de los tokens `low`, `medium` y `high`.

Esto no es todavía nuestro clasificador entrenado: es el punto de partida que representa el conocimiento lingüístico general del modelo.

También calculamos una baseline de control de **clase mayoritaria**.


In [ ]:
from transformers import AutoModelForMaskedLM

# Baseline zero-shot: mismo modelo base, sin fine-tuning.
mlm_model = AutoModelForMaskedLM.from_pretrained(BASE_MODEL).to(device)
mlm_model.eval()

mask_token = tokenizer.mask_token
candidate_labels = ["low", "medium", "high"]

candidate_ids = {}
for label in candidate_labels:
    ids = tokenizer(label, add_special_tokens=False)["input_ids"]
    if len(ids) != 1:
        raise ValueError(
            f"'{label}' no es un único token en este tokenizer: {ids}. "
            "Cambiar la estrategia zero-shot."
        )
    candidate_ids[label.upper()] = ids[0]

print("Candidate token ids:", candidate_ids)


In [ ]:
@torch.no_grad()
def zero_shot_predict(text):
    prompt = f"{text} The food waste level is {mask_token}."
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    logits = mlm_model(**inputs).logits[0]

    mask_positions = (inputs["input_ids"][0] == tokenizer.mask_token_id).nonzero(as_tuple=True)[0]
    if len(mask_positions) != 1:
        raise ValueError("La plantilla debe contener exactamente un [MASK].")

    mask_logits = logits[mask_positions.item()]
    candidate_logits = torch.tensor(
        [mask_logits[candidate_ids[label]] for label in ["LOW", "MEDIUM", "HIGH"]],
        device=device,
    )

    probs = torch.softmax(candidate_logits, dim=0)
    pred_idx = int(torch.argmax(probs).item())
    return ["LOW", "MEDIUM", "HIGH"][pred_idx]

# Probamos antes de evaluar todo.
print(zero_shot_predict(val_df.loc[0, "text"]))


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix

# Zero-shot baseline
baseline_zero_preds = [zero_shot_predict(text) for text in val_df["text"]]
y_val = val_df["label"].tolist()

baseline_zero = {
    "accuracy": accuracy_score(y_val, baseline_zero_preds),
    "precision_macro": precision_score(y_val, baseline_zero_preds, average="macro", zero_division=0),
    "recall_macro": recall_score(y_val, baseline_zero_preds, average="macro", zero_division=0),
    "f1_macro": f1_score(y_val, baseline_zero_preds, average="macro"),
}

# Baseline de clase mayoritaria
majority_label = train_df["label"].value_counts().idxmax()
majority_preds = [majority_label] * len(val_df)

baseline_majority = {
    "accuracy": accuracy_score(y_val, majority_preds),
    "precision_macro": precision_score(y_val, majority_preds, average="macro", zero_division=0),
    "recall_macro": recall_score(y_val, majority_preds, average="macro", zero_division=0),
    "f1_macro": f1_score(y_val, majority_preds, average="macro"),
}

print("Baseline zero-shot:")
display(pd.DataFrame([baseline_zero], index=["DistilBERT zero-shot"]).round(4))

print("Baseline clase mayoritaria:")
display(pd.DataFrame([baseline_majority], index=[f"Majority ({majority_label})"]).round(4))


## 7 · Dataset Hugging Face y tokenización

Ahora preparamos el dataset para `Trainer`.

Usamos `DataCollatorWithPadding` para realizar padding dinámico y no agregar tokens innecesarios.


In [ ]:
from datasets import Dataset
from transformers import DataCollatorWithPadding

train_hf = Dataset.from_pandas(
    train_df.assign(labels=train_df["label"].map(label2id)).drop(columns=["label"])
)
val_hf = Dataset.from_pandas(
    val_df.assign(labels=val_df["label"].map(label2id)).drop(columns=["label"])
)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=256,
    )

train_tok = train_hf.map(tokenize_batch, batched=True)
val_tok = val_hf.map(tokenize_batch, batched=True)

columns_to_keep = ["input_ids", "attention_mask", "labels"]
train_tok = train_tok.remove_columns(
    [c for c in train_tok.column_names if c not in columns_to_keep]
)
val_tok = val_tok.remove_columns(
    [c for c in val_tok.column_names if c not in columns_to_keep]
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(train_tok)
print(val_tok)


## 8 · Modelo de clasificación + LoRA

La asignación exige fine-tuning eficiente, no full fine-tuning.

Configuración inicial:

- `r = 8`
- `alpha = 16`
- `dropout = 0.10`
- `target_modules = ["q_lin", "v_lin"]`

En DistilBERT, `q_lin` y `v_lin` son las proyecciones de query y value de la atención.

Además guardamos `pre_classifier` y `classifier`, porque la cabeza de clasificación para nuestras tres clases se inicializa al crear el modelo y debe entrenarse.


In [ ]:
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, TaskType, get_peft_model

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.10,
    target_modules=["q_lin", "v_lin"],
    modules_to_save=["pre_classifier", "classifier"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## 9 · Métricas

La métrica principal será **F1-macro**, porque queremos dar el mismo peso a LOW, MEDIUM y HIGH.

También reportaremos Accuracy, Precision-macro y Recall-macro.


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision_macro": precision_score(labels, predictions, average="macro", zero_division=0),
        "recall_macro": recall_score(labels, predictions, average="macro", zero_division=0),
        "f1_macro": f1_score(labels, predictions, average="macro"),
    }


## 10 · Fine-tuning con Trainer

La estructura sigue la asignación M1: entrenamiento con `Trainer`, evaluación durante el proceso y conservación de los outputs.

Los hiperparámetros son deliberadamente modestos para que el experimento sea viable en una T4 gratuita. Si el entrenamiento tarda demasiado, se puede reducir el número de épocas después de registrar el experimento.


In [ ]:
from transformers import TrainingArguments, Trainer
import inspect

OUTPUT_DIR = "./distilbert-lora-food-waste"

# Compatibilidad entre versiones de transformers:
ta_params = inspect.signature(TrainingArguments.__init__).parameters

training_kwargs = dict(
    output_dir=OUTPUT_DIR,
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=20,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    report_to="none",
    seed=SEED,
)

if "eval_strategy" in ta_params:
    training_kwargs["eval_strategy"] = "epoch"
    training_kwargs["save_strategy"] = "epoch"
else:
    training_kwargs["evaluation_strategy"] = "epoch"
    training_kwargs["save_strategy"] = "epoch"

training_args = TrainingArguments(**training_kwargs)

trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer_params = inspect.signature(Trainer.__init__).parameters
if "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)

print("Trainer listo.")


In [ ]:
# Entrenamiento
train_result = trainer.train()

print("\nEntrenamiento terminado.")
print(train_result.metrics)


## 11 · Evaluación final

Evaluamos el modelo afinado en el **mismo validation set** usado por los baselines.


In [ ]:
fine_tuned_metrics = trainer.evaluate()

print("Métricas del modelo afinado:")
display(pd.DataFrame([fine_tuned_metrics]).round(4))


In [ ]:
# Predicciones y métricas detalladas
pred_output = trainer.predict(val_tok)

fine_preds_ids = np.argmax(pred_output.predictions, axis=-1)
fine_preds = [id2label[int(i)] for i in fine_preds_ids]

fine_metrics = {
    "accuracy": accuracy_score(y_val, fine_preds),
    "precision_macro": precision_score(y_val, fine_preds, average="macro", zero_division=0),
    "recall_macro": recall_score(y_val, fine_preds, average="macro", zero_division=0),
    "f1_macro": f1_score(y_val, fine_preds, average="macro"),
}

display(pd.DataFrame([
    baseline_zero,
    fine_metrics
], index=[
    "DistilBERT zero-shot",
    "DistilBERT + LoRA"
]).round(4))


## 12 · Comparación con baseline

La pregunta de M1 no es solamente "¿qué número obtuvo el modelo?", sino:

> **¿qué aportó el fine-tuning?**

Calculamos el delta de F1-macro respecto al baseline zero-shot.


In [ ]:
comparison = pd.DataFrame([
    {
        "model": "DistilBERT zero-shot",
        "accuracy": baseline_zero["accuracy"],
        "precision_macro": baseline_zero["precision_macro"],
        "recall_macro": baseline_zero["recall_macro"],
        "f1_macro": baseline_zero["f1_macro"],
    },
    {
        "model": "DistilBERT + LoRA",
        "accuracy": fine_metrics["accuracy"],
        "precision_macro": fine_metrics["precision_macro"],
        "recall_macro": fine_metrics["recall_macro"],
        "f1_macro": fine_metrics["f1_macro"],
    },
])

comparison["delta_f1_vs_zero_shot"] = comparison["f1_macro"] - baseline_zero["f1_macro"]
display(comparison.round(4))

delta = fine_metrics["f1_macro"] - baseline_zero["f1_macro"]
print(f"Delta F1-macro del fine-tuning frente al baseline: {delta:+.4f}")


## 13 · F1 por clase y matriz de confusión

El F1-macro resume el desempeño global, pero la matriz de confusión permite ver si una clase concreta sigue siendo difícil.


In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

print(classification_report(
    y_val,
    fine_preds,
    labels=label_order,
    zero_division=0
))

cm = confusion_matrix(y_val, fine_preds, labels=label_order)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_order)
disp.plot(ax=ax, values_format="d", cmap="Blues", colorbar=False)
ax.set_title("DistilBERT + LoRA — Validation confusion matrix")
plt.show()


## 14 · Ejemplos cualitativos

La asignación exige al menos tres ejemplos de entrada → salida.

Mostramos ejemplos reales del validation set junto con la etiqueta verdadera y la predicción.


In [ ]:
examples = val_df.copy()
examples["prediction"] = fine_preds

# Preferimos ejemplos correctos si existen; si no, mostramos los primeros.
correct = examples[examples["label"] == examples["prediction"]]
wrong = examples[examples["label"] != examples["prediction"]]

print("### Ejemplos correctos")
display(correct[["text", "label", "prediction"]].head(3))

print("\n### Ejemplos donde el modelo falla")
display(wrong[["text", "label", "prediction"]].head(3))


## 15 · Conclusiones

Completar esta celda **después de ejecutar el notebook**.

La conclusión debe responder:

1. ¿El modelo `DistilBERT + LoRA` superó al baseline?
2. ¿Cuánto cambió el F1-macro?
3. ¿Qué clase fue más difícil?
4. ¿Qué limitaciones del dataset pueden explicar los errores?
5. ¿Qué cambiarían en una siguiente iteración?

No inventar resultados antes de ejecutar el experimento.


In [ ]:
# Resumen automático para facilitar la redacción del informe
best_row = comparison.loc[comparison["f1_macro"].idxmax()]

print(f"Mejor modelo según F1-macro: {best_row['model']}")
print(f"F1-macro: {best_row['f1_macro']:.4f}")
print(f"F1-macro baseline zero-shot: {baseline_zero['f1_macro']:.4f}")
print(f"Delta: {delta:+.4f}")

if delta > 0:
    print("Lectura inicial: el fine-tuning mejoró el F1-macro frente al baseline.")
else:
    print("Lectura inicial: el fine-tuning no superó el baseline; revisar hiperparámetros, representación textual y dataset.")


## 16 · Checklist de entrega M1

- [ ] Notebook corre de principio a fin en Colab.
- [ ] Modelo y tokenizer cargados desde Hugging Face.
- [ ] Dataset del dominio documentado.
- [ ] Split train/validation estratificado.
- [ ] `Wastage Food Amount` no entra como feature.
- [ ] Baseline explícito.
- [ ] LoRA configurado: rank, alpha, target modules.
- [ ] Trainer API utilizada.
- [ ] Outputs del entrenamiento conservados.
- [ ] Métrica principal: F1-macro.
- [ ] Comparación sobre el mismo validation set.
- [ ] Matriz de confusión.
- [ ] Al menos tres ejemplos cualitativos.
- [ ] Semilla fijada.
- [ ] README/MD con fuente, tamaño, idioma, licencia, tarea y limitaciones.
- [ ] No se reportan números inventados: las métricas finales salen de la ejecución del notebook.
